# Kotoba TTS - Real-time Text-to-Speech with Bidirectional Streaming

This notebook demonstrates how to use the Kotoba TTS (Text-to-Speech) model package from AWS Marketplace.

**Product**: Kotoba TTS - Real-time Japanese Speech Synthesis  
**Seller**: Kotoba AI

## Key Features

- Real-time streaming synthesis with sub-second time-to-first-audio
- Preset Japanese voices (English available where the deployment enables it via `SUPPORTED_LANGUAGES`)
- 24 kHz mono `pcm_f32` audio output (little-endian float32)
- Scalable GPU-accelerated inference

---

## IMPORTANT: Bidirectional Streaming Only

This product uses **SageMaker bidirectional streaming** exclusively:

- Standard `POST /invocations` is **NOT supported**
- Batch transform jobs are **NOT supported**
- Requires **Python 3.12+** with `aws-sdk-python[sagemaker-runtime-http2]`

---

## Prerequisites

1. **AWS Account** with SageMaker permissions
2. **Python 3.12+** (required for HTTP/2 SDK)
3. **AWS Marketplace subscription** to Kotoba TTS

### Required IAM Permissions

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "sagemaker:CreateEndpoint",
                "sagemaker:CreateEndpointConfig",
                "sagemaker:DeleteEndpoint",
                "sagemaker:DeleteEndpointConfig",
                "sagemaker:DescribeEndpoint",
                "sagemaker:InvokeEndpoint",
                "sagemaker:InvokeEndpointWithResponseStream"
            ],
            "Resource": "*"
        }
    ]
}
```

## Table of Contents

1. [Subscribe to Model Package](#1.-Subscribe-to-Model-Package)
2. [Setup](#2.-Setup)
3. [Deploy Real-time Endpoint](#3.-Deploy-Real-time-Endpoint)
4. [Perform Real-time Synthesis](#4.-Perform-Real-time-Synthesis)
5. [Troubleshooting](#5.-Troubleshooting)
6. [Clean-up](#6.-Clean-up)

## 1. Subscribe to Model Package

To subscribe to the Kotoba TTS model package:

1. Open the [Kotoba TTS listing on AWS Marketplace](#) <!-- TODO: Update with actual URL -->
2. Click **Continue to Subscribe**
3. Review the pricing and EULA, then click **Subscribe**
4. Once subscribed, click **Continue to Configuration**
5. Select your preferred region and note the **Model Package ARN**

## 2. Setup

### 2.1 Install Dependencies

Run this cell first — the Python version check in 2.2 doesn't require these packages, but every cell after that does.

In [ ]:
!pip install -q --upgrade boto3 numpy
!pip install -q --upgrade "sagemaker>=2.0,<3"
!pip install -q --upgrade "aws-sdk-python[sagemaker-runtime-http2]"


### 2.2 Check Python Version

`aws-sdk-python` requires Python 3.12+. If this cell raises, switch your kernel (or create a new environment) before continuing.

In [ ]:
import sys

if sys.version_info < (3, 12):
    raise RuntimeError(
        f"Python 3.12+ is required for SageMaker bidirectional streaming. "
        f"Current version: {sys.version}"
    )

print(f"Python version: {sys.version}")


### 2.3 Import Libraries

In [ ]:
import asyncio
import base64
import json
import time
import wave

import boto3
import numpy as np
import sagemaker

# SageMaker HTTP/2 SDK for bidirectional streaming
from aws_sdk_sagemaker_runtime_http2.client import SageMakerRuntimeHTTP2Client
from aws_sdk_sagemaker_runtime_http2.config import Config, HTTPAuthSchemeResolver
from aws_sdk_sagemaker_runtime_http2.models import (
    InvokeEndpointWithBidirectionalStreamInput,
    RequestPayloadPart,
    RequestStreamEventPayloadPart,
)
from sagemaker import ModelPackage
from smithy_aws_core.auth.sigv4 import SigV4AuthScheme
from smithy_aws_core.identity import StaticCredentialsResolver

print("All dependencies imported successfully.")

### 2.4 Configure AWS Session

SageMaker needs an **execution role** (an IAM *role*, not an IAM *user*) to assume when creating the endpoint.

- **On SageMaker Studio / Notebook Instance**: leave `SAGEMAKER_ROLE_ARN = None`. `sagemaker.get_execution_role()` will pick up the attached role automatically.
- **Running locally (IAM user / CLI profile)**: set `SAGEMAKER_ROLE_ARN` to the ARN of a role whose trust policy allows `sagemaker.amazonaws.com` to assume it and which has (at minimum) the `AmazonSageMakerFullAccess` managed policy.


In [ ]:
# --- Execution role -----------------------------------------------------
# If running on SageMaker Studio or a Notebook Instance, leave this as None.
# If running locally as an IAM user, set it to the ARN of a SageMaker-capable role, e.g.
#   SAGEMAKER_ROLE_ARN = "arn:aws:iam::123456789012:role/SageMakerExecutionRole"
SAGEMAKER_ROLE_ARN: str | None = None
# ------------------------------------------------------------------------

sagemaker_session = sagemaker.Session()
region = sagemaker_session.boto_region_name

if SAGEMAKER_ROLE_ARN is not None:
    role = SAGEMAKER_ROLE_ARN
else:
    try:
        role = sagemaker.get_execution_role()
    except ValueError as exc:
        raise ValueError(
            "get_execution_role() only works in SageMaker-managed environments. "
            "Set SAGEMAKER_ROLE_ARN above to a SageMaker execution role ARN "
            "(not your IAM user ARN). See the markdown cell above for role requirements."
        ) from exc

print(f"Region: {region}")
print(f"Role:   {role}")


### 2.5 Configure Model Package ARN

Replace `<MODEL_PACKAGE_ARN>` with the ARN from your AWS Marketplace subscription.

In [ ]:
# TODO: Replace with your Model Package ARN from AWS Marketplace
MODEL_PACKAGE_ARN = "<MODEL_PACKAGE_ARN>"

if MODEL_PACKAGE_ARN.startswith("<"):
    raise ValueError(
        "Please replace MODEL_PACKAGE_ARN with your actual Model Package ARN "
        "from AWS Marketplace subscription."
    )


## 3. Deploy Real-time Endpoint

### 3.1 Create Endpoint

Supported instance types:

| Instance | vCPU | Memory | GPU | Notes |
|----------|------|--------|-----|-------|
| `ml.g6.2xlarge` | 8  | 32 GiB | 1x NVIDIA L4 | Default, good balance of cost and throughput |
| `ml.g6.4xlarge` | 16 | 64 GiB | 1x NVIDIA L4 | Choose for higher concurrency |


In [ ]:
# Endpoint configuration
ENDPOINT_NAME = f"kotoba-tts-{int(time.time())}"
INSTANCE_TYPE = "ml.g6.2xlarge"  # or "ml.g6.4xlarge"

print(f"Endpoint name: {ENDPOINT_NAME}")
print(f"Instance type: {INSTANCE_TYPE}")


In [ ]:
# Create Model from Model Package
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=sagemaker_session,
)

# Deploy endpoint (this takes 5-10 minutes)
print(f"Deploying endpoint '{ENDPOINT_NAME}'...")

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)

print(f"Endpoint '{ENDPOINT_NAME}' is InService.")


## 4. Perform Real-time Synthesis

### 4.1 Connection Protocol

Kotoba TTS uses SageMaker bidirectional streaming:

```
Client (HTTP/2 SDK) -> SageMaker Runtime (HTTPS:8443) -> Sidecar -> Container (WS:8080)
```

**Note**: Standard WebSocket libraries cannot connect directly to SageMaker endpoints.

### 4.2 Output Audio Format

The server emits a single hardcoded audio format. Audio is base64-encoded
inside `audio.chunk.audio`.

| Format    | Sample Rate | Channels | Encoding              |
| --------- | ----------- | -------- | --------------------- |
| `pcm_f32` | 24000 Hz    | 1 (mono) | Little-endian float32 |

### 4.3 Protocol Flow

The client must send `open` first; the server replies with `session.created`
once the language and speaker are validated. If `open` is not sent within
`WS_OPEN_TIMEOUT` (default 10s), the server closes the connection.

```
Client                                    Server
  |---- open ------------------------------>|
  |<---- session.created -------------------|
  |---- response.create ------------------->|
  |<---- response.created ------------------|
  |---- input_text_buffer.append ---------->| (concurrent)
  |<---- audio.chunk (isFinal:false) -------|
  |---- input_text_buffer.commit ---------->|
  |<---- audio.chunk (isFinal:true) --------|
  |<---- response.done ---------------------|
```

### 4.4 Event Reference

#### Client Events (you send)

| Event                         | Description                                             |
| ----------------------------- | ------------------------------------------------------- |
| `open`                        | First message; configure language / speaker_id          |
| `response.create`             | Start a synthesis turn (repeatable per session)         |
| `input_text_buffer.append`    | Send a text chunk (sentence-level recommended)          |
| `input_text_buffer.commit`    | Signal end of text input for this turn                  |
| `response.cancel`             | Cancel the currently active synthesis                   |

#### Server Events (you receive)

| Event                         | Description                                             |
| ----------------------------- | ------------------------------------------------------- |
| `session.created`             | Session established (format, sample_rate, channels)     |
| `response.created`            | Synthesis turn started                                  |
| `audio.chunk`                 | Base64 audio delta (`isFinal` marks the last chunk)     |
| `response.done`               | Turn finished (`status`: completed / cancelled / failed)|
| `error`                       | Fatal error (connection will close)                     |
| `timeout`                     | Worker result timeout notification                      |

See [docs/tts/data/sample_input.json](./data/sample_input.json) and [docs/tts/data/sample_output.json](./data/sample_output.json) for canonical event payloads.

### 4.5 Prepare Input Text

In [ ]:
SAMPLE_RATE = 24000
LANGUAGE = "ja"
# `default` is currently the only supported preset speaker.
SPEAKER_ID = "default"

# Input text to synthesize (split into sentence-level chunks for streaming).
TEXT_CHUNKS = [
    "こんにちは、",
    "世界。",
    "今日もよろしくお願いします。",
]

full_text = "".join(TEXT_CHUNKS)
print(f"Language: {LANGUAGE}")
print(f"Speaker: {SPEAKER_ID}")
print(f"Text: {full_text!r}")
print(f"Chunks: {len(TEXT_CHUNKS)}")


### 4.6 Streaming Inference

In [ ]:
async def synthesize_text(
    endpoint_name: str,
    text_chunks: list[str],
    language: str = "ja",
    speaker_id: str = "default",
) -> bytes:
    """
    Synthesize speech using Kotoba TTS bidirectional streaming.

    Args:
        endpoint_name: SageMaker endpoint name
        text_chunks: list of text fragments to stream
        language: target language (must be in the deployment's `SUPPORTED_LANGUAGES`)
        speaker_id: preset voice identifier (currently only `default` is supported)

    Returns:
        Raw audio bytes (pcm_f32, 24000 Hz, mono)
    """
    # Get AWS credentials
    session = boto3.Session()
    credentials = session.get_credentials().get_frozen_credentials()

    # Configure HTTP/2 client
    config = Config(
        endpoint_uri=f"https://runtime.sagemaker.{region}.amazonaws.com:8443",
        region=region,
        aws_access_key_id=credentials.access_key,
        aws_secret_access_key=credentials.secret_key,
        aws_session_token=credentials.token,
        aws_credentials_identity_resolver=StaticCredentialsResolver(),
        auth_scheme_resolver=HTTPAuthSchemeResolver(),
        auth_schemes={"aws.auth#sigv4": SigV4AuthScheme(service="sagemaker")},
    )

    client = SageMakerRuntimeHTTP2Client(config=config)

    # Start bidirectional stream
    stream = await client.invoke_endpoint_with_bidirectional_stream(
        InvokeEndpointWithBidirectionalStreamInput(
            endpoint_name=endpoint_name,
            model_invocation_path="invocations-bidirectional-stream",
        )
    )

    async def send_event(payload: dict):
        body = json.dumps(payload).encode("utf-8")
        # NOTE: data_type="UTF8" is required so SageMaker forwards the frame
        # as text. Without it the server (which calls ws.receive_text()) cannot
        # decode the payload and the connection fails before any handler runs.
        request = RequestStreamEventPayloadPart(
            value=RequestPayloadPart(bytes_=body, data_type="UTF8")
        )
        await stream.input_stream.send(request)

    output = await stream.await_output()
    output_stream = output[1] if isinstance(output, tuple) else output

    # 1) Send open first. The server waits for this within WS_OPEN_TIMEOUT (~10s)
    #    and replies with session.created once language/speaker are validated.
    await send_event({
        "type": "open",
        "language": language,
        "speaker_id": speaker_id,
    })

    # 2) Receive session.created
    message = await output_stream.receive()
    data = json.loads(message.value.bytes_.decode())
    assert data["type"] == "session.created", data
    fmt = data.get("format")
    sr = data.get("sample_rate")
    ch = data.get("channels")
    print(f"Session created: format={fmt}, sr={sr}, ch={ch}")

    # 3) Start a synthesis turn
    await send_event({"type": "response.create"})
    message = await output_stream.receive()
    data = json.loads(message.value.bytes_.decode())
    assert data["type"] == "response.created", data
    print("Response started")

    audio_buffer = bytearray()

    async def send_text():
        for chunk in text_chunks:
            await send_event({
                "type": "input_text_buffer.append",
                "text": chunk,
            })
        await send_event({"type": "input_text_buffer.commit"})

    async def receive_audio():
        while True:
            message = await output_stream.receive()
            if message is None:
                break
            data = json.loads(message.value.bytes_.decode())
            event_type = data["type"]
            if event_type == "audio.chunk":
                audio_b64 = data.get("audio", "")
                is_final = data.get("isFinal", False)
                audio_buffer.extend(base64.b64decode(audio_b64))
                print(f"Got audio chunk ({len(audio_b64)} b64 chars, isFinal={is_final})")
                if is_final:
                    continue
            elif event_type == "response.done":
                status = data.get("response", {}).get("status")
                print(f"Response done: status={status}")
                if status != "completed":
                    raise RuntimeError(f"Synthesis failed: {data}")
                break
            elif event_type in ("error", "timeout"):
                raise RuntimeError(f"Server event: {data}")

    await asyncio.gather(send_text(), receive_audio())
    await stream.input_stream.close()

    return bytes(audio_buffer)


In [ ]:
# Run synthesis
print("Starting synthesis...\n")

audio_bytes = await synthesize_text(
    endpoint_name=ENDPOINT_NAME,
    text_chunks=TEXT_CHUNKS,
    language=LANGUAGE,
    speaker_id=SPEAKER_ID,
)

print(f"\nReceived {len(audio_bytes)} bytes of pcm_f32 audio")

# Convert pcm_f32 -> int16 and save as WAV for easy playback
samples_f32 = np.frombuffer(audio_bytes, dtype=np.float32)
samples_i16 = np.clip(samples_f32, -1.0, 1.0) * 32767.0
samples_i16 = samples_i16.astype(np.int16)

out_path = "sample_output.wav"
with wave.open(out_path, "wb") as w:
    w.setnchannels(1)
    w.setsampwidth(2)
    w.setframerate(SAMPLE_RATE)
    w.writeframes(samples_i16.tobytes())

print(f"Saved synthesized audio to {out_path} ({len(samples_i16) / SAMPLE_RATE:.2f} s)")

## 5. Troubleshooting

| Symptom | Likely cause & fix |
|---------|--------------------|
| `RuntimeError: Python 3.12+ is required` | The HTTP/2 SDK requires Python 3.12+. Switch your kernel or create a new conda/venv with Python 3.12. |
| `ModuleNotFoundError: aws_sdk_sagemaker_runtime_http2` | Re-run the install cell in 2.1 (`pip install aws-sdk-python[sagemaker-runtime-http2]`). The package name uses underscores when imported. |
| Connection closes immediately after invoke (no `session.created`) | The first event must be `open` and must be sent within `WS_OPEN_TIMEOUT` (default 10s). Make sure `RequestPayloadPart` is constructed with `data_type="UTF8"` so SageMaker forwards a text frame. |
| Server replies with `error` / `unsupported_language` | The deployment's `SUPPORTED_LANGUAGES` does not include the requested language. Use `language="ja"` or redeploy with a wider language list. |
| `response.done` with `status: failed` and `unknown_speaker_id` | Currently only `speaker_id="default"` is supported. Other preset keys may exist in the runtime code but are not part of the shipped Marketplace bundle. |
| `timeout` event during synthesis | The worker did not produce audio within `WS_RESULT_TIMEOUT` (default 60s). Usually transient — retry the response. If it persists, check the endpoint's CloudWatch logs. |
| Idle connection closed after ~60s | Hit `WS_IDLE_TIMEOUT`. Send a new `response.create` (or any client event) before the timeout, or open a fresh connection per turn. |
| `AccessDeniedException` at `model.deploy(...)` | The execution role cannot pull the Marketplace model. Verify the AWS Marketplace subscription is active and that the role has `AmazonSageMakerFullAccess` (or equivalent). |
| Saved WAV is silent / sounds like white noise | The pcm_f32 -> int16 conversion was skipped or the byte order is wrong. The server emits little-endian float32; decode with `np.frombuffer(..., dtype=np.float32)` before scaling. |

## 6. Clean-up

**Important**: Delete the endpoint when done. GPU instances (`ml.g6.2xlarge` / `ml.g6.4xlarge`) are billed **per hour while the endpoint is active**, even when idle.

In [ ]:
print(f"Deleting endpoint '{ENDPOINT_NAME}'...")

sagemaker_session.delete_endpoint(ENDPOINT_NAME)
sagemaker_session.delete_endpoint_config(ENDPOINT_NAME)

print("Endpoint deleted.")


### Unsubscribe from Model Package (Optional)

To unsubscribe from the Kotoba TTS model package:

1. Go to [AWS Marketplace Subscriptions](https://console.aws.amazon.com/marketplace/home#/subscriptions)
2. Find "Kotoba TTS" in your subscriptions
3. Click **Manage** -> **Actions** -> **Cancel subscription**

**Note**: You must delete all endpoints using this model package before unsubscribing.

---

## Support

For technical support, please contact Kotoba AI support.

## References

- Canonical protocol samples: [docs/tts/data/sample_input.json](./data/sample_input.json), [docs/tts/data/sample_output.json](./data/sample_output.json), [docs/tts/data/README.md](./data/README.md)
- [SageMaker Bidirectional Streaming Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints-test-endpoints.html)
- [AWS SDK for Python (Boto3)](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)